In [ ]:
import streamlit as st
import pandas as pd
import psycopg2
from PIL import Image
import requests
from io import BytesIO
import base64

# Set page to full width and add logo
st.set_page_config(page_title="RED BUS.IN", layout="wide")

# Add logo to top left corner
logo_url = "https://www.redbus.in/i/59538b35953097248522a65b4b79650e.png"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(logo_url, headers=headers)
logo = Image.open(BytesIO(response.content))

# Resize the logo
logo = logo.resize((150, 50))

# Create a container for the logo and title
col1, col2 = st.columns([1, 5])
with col1:
    st.image(logo)
with col2:
    st.markdown("<h1 style='color: red;'>RedBus Data Booking</h1>", unsafe_allow_html=True)

# PostgreSQL connection details
DB_CONFIG = {
    "dbname": "red_bus",
    "user": "postgres",
    "password": "sample12",
    "host": "localhost",
    "port": "5432",
}

# Function to fetch data from PostgreSQL
@st.cache_data
def get_data():
    conn = psycopg2.connect(**DB_CONFIG)
    query = "SELECT * FROM bus_routes"  
    df = pd.read_sql(query, conn)
    conn.close()
    
    # Ensure 'bustype' is a string and strip spaces
    if "bustype" in df.columns:
        df["bustype"] = df["bustype"].astype(str).str.strip()
    
    # Ensure 'price' is numeric
    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
    
    return df

# Load Data
df = get_data()

# Sidebar filters
st.sidebar.header("Filter Options")

# Multiselect for Route Name
route_names = df["route_name"].unique().tolist()
selected_routes = st.sidebar.multiselect(
    "Select Route Name(s)", 
    route_names,
    default=route_names[:1] if route_names else None
)

# Apply route filter
if len(selected_routes) > 0:
    filtered_df = df[df["route_name"].isin(selected_routes)]
else:
    filtered_df = df.copy()

# Multiselect for Star Rating (if applicable)
if "star_rating" in df.columns and not filtered_df.empty:
    star_ratings = sorted(filtered_df["star_rating"].dropna().unique().tolist())
    if star_ratings:
        selected_star_ratings = st.sidebar.multiselect(
            "Select Star Rating(s)", 
            star_ratings,
            default=star_ratings[:1] if star_ratings else None
        )
        if len(selected_star_ratings) > 0:
            filtered_df = filtered_df[filtered_df["star_rating"].isin(selected_star_ratings)]

# Multiselect for Bus Type (if applicable)
if "bus_type" in df.columns and not filtered_df.empty:  
    bustypes = sorted(filtered_df["bus_type"].dropna().unique().tolist())
    if bustypes:
        selected_bustypes = st.sidebar.multiselect(
            "Select Bus Type(s)",
            bustypes,
            default=bustypes[:1] if bustypes else None
        )
        if len(selected_bustypes) > 0:
            filtered_df = filtered_df[filtered_df["bus_type"].isin(selected_bustypes)]

# Price Slider (if applicable)
if "price" in filtered_df.columns and not filtered_df["price"].dropna().empty:
    min_price = int(filtered_df["price"].min())
    max_price = int(filtered_df["price"].max())

    if min_price == max_price:
        min_price = max(0, min_price - 100)
        max_price = min_price + 200

    selected_price_range = st.sidebar.slider(
        "Select Price Range", min_price, max_price, (min_price, max_price)
    )
    filtered_df = filtered_df[
        (filtered_df["price"] >= selected_price_range[0]) & 
        (filtered_df["price"] <= selected_price_range[1])
    ]

# Column Selection
selected_columns = st.sidebar.multiselect(
    "Select columns to view", 
    df.columns, 
    default=df.columns
)

# Display Data
st.write(f"### Filtered Bus Data")
st.dataframe(filtered_df[selected_columns], use_container_width=True)